<a href="https://colab.research.google.com/github/aMDy0k/workspace/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###### import

In [163]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sympy import *
import requests
import json
import time
import os
import ast

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


###### словарь

In [164]:
heroes = {
    1: "Anti-Mage", 2: "Axe", 3: "Bane", 4: "Bloodseeker", 5: "Crystal Maiden",
    6: "Drow Ranger", 7: "Earthshaker", 8: "Juggernaut", 9: "Mirana", 10: "Morphling",
    11: "Shadow Fiend", 12: "Phantom Lancer", 13: "Puck", 14: "Pudge", 15: "Razor",
    16: "Sand King", 17: "Storm Spirit", 18: "Sven", 19: "Tiny", 20: "Vengeful Spirit",
    21: "Windranger", 22: "Zeus", 23: "Kunkka", 25: "Lina", 26: "Lion",
    27: "Shadow Shaman", 28: "Slardar", 29: "Tidehunter", 30: "Witch Doctor", 31: "Lich",
    32: "Riki", 33: "Enigma", 34: "Tinker", 35: "Sniper", 36: "Necrophos",
    37: "Warlock", 38: "Beastmaster", 39: "Queen of Pain", 40: "Venomancer", 41: "Faceless Void",
    42: "Wraith King", 43: "Death Prophet", 44: "Phantom Assassin", 45: "Pugna", 46: "Templar Assassin",
    47: "Viper", 48: "Luna", 49: "Dragon Knight", 50: "Dazzle", 51: "Clockwerk",
    52: "Leshrac", 53: "Nature's Prophet", 54: "Lifestealer", 55: "Dark Seer", 56: "Clinkz",
    57: "Omniknight", 58: "Enchantress", 59: "Huskar", 60: "Night Stalker", 61: "Broodmother",
    62: "Bounty Hunter", 63: "Weaver", 64: "Jakiro", 65: "Batrider", 66: "Chen",
    67: "Spectre", 68: "Ancient Apparition", 69: "Doom", 70: "Ursa", 71: "Spirit Breaker",
    72: "Gyrocopter", 73: "Alchemist", 74: "Invoker", 75: "Silencer", 76: "Outworld Destroyer",
    77: "Lycan", 78: "Brewmaster", 79: "Shadow Demon", 80: "Lone Druid", 81: "Chaos Knight",
    82: "Meepo", 83: "Treant Protector", 84: "Ogre Magi", 85: "Undying", 86: "Rubick",
    87: "Disruptor", 88: "Nyx Assassin", 89: "Naga Siren", 90: "Keeper of the Light", 91: "Io",
    92: "Visage", 93: "Slark", 94: "Medusa", 95: "Troll Warlord", 96: "Centaur Warrunner",
    97: "Magnus", 98: "Timbersaw", 99: "Bristleback", 100: "Tusk", 101: "Skywrath Mage",
    102: "Abaddon", 103: "Elder Titan", 104: "Legion Commander", 105: "Techies", 106: "Ember Spirit",
    107: "Earth Spirit", 108: "Underlord", 109: "Terrorblade", 110: "Phoenix", 111: "Oracle",
    112: "Winter Wyvern", 113: "Arc Warden", 114: "Monkey King", 119: "Dark Willow", 120: "Pangolier",
    121: "Grimstroke", 123: "Hoodwink", 126: "Void Spirit", 128: "Snapfire", 129: "Mars",
    131: "Muerta", 135: "Dawnbreaker", 136: "Marci", 137: "Primal Beast",
    138: "Ringmaster", 145: "Kez", 155: "Largo"
}

### data

In [170]:
    def clean_teams_in_row(row):
        def parse_value(val):
            if isinstance(val, str):
                try:
                    return json.loads(val)
                except:
                    try:
                        return ast.literal_eval(val)
                    except:
                        return [
                            int(x)
                            for x in val.split(",")
                            if x.strip().isdigit()
                        ]
            return [int(x) for x in val] if isinstance(val, list) else []

        row["radiant_team"] = parse_value(row.get("radiant_team", []))
        row["dire_team"] = parse_value(row.get("dire_team", []))
        return row

In [172]:
def parser():
    global df

    # Проверяем, существует ли глобальный df в памяти ноутбука
    if "df" in globals() and not df.empty:
        # Так как match_id теперь в индексе, берем минимум прямо из индекса
        min_id = df.index.min()

        # Проверяем, что минимум — это реальное число
        if pd.notna(min_id) and min_id > 1000000000:
            less_than_match_id = int(min_id)
            print(
                f"Парсер продолжит сбор, начиная с match_id < {less_than_match_id}"
            )
        else:
            less_than_match_id = None
            print("Парсер начинает сбор с самых актуальных матчей.")
    else:
        less_than_match_id = None
        print("Парсер начинает сбор с самых актуальных матчей.")

    # ---------------------------

    # Базовый URL для получения СЫРЫХ публичных матчей
    url = "https://api.opendota.com/api/publicMatches"
    all_raw_matches = []

    print("Начинаем выкачивать сырой датасет напрямую...")

    for i in range(50):
        params = {}
        if less_than_match_id:
            params["less_than_match_id"] = less_than_match_id

        try:
            response = requests.get(url, params=params)

            if response.status_code == 200:
                batch = response.json()
                if not batch:
                    print("Матчи закончились.")
                    break

                all_raw_matches.extend(batch)
                print(
                    f"Скачана пачка {i+1}. Всего матчей в датасете: {len(all_raw_matches)}"
                )

                less_than_match_id = batch[-1]["match_id"]
                time.sleep(1)
            else:
                print(
                    f"Ошибка. Статус: {response.status_code}, Ответ: {response.text}"
                )
                break
        except Exception as e:
            print(f"Ошибка сети: {e}")
            break

    if all_raw_matches:
        filename = "raw_matches_dataset.json"
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(all_raw_matches, f, indent=4, ensure_ascii=False)

        print(f"\nСырой датасет сохранен в файл: '{filename}'")
    else:
        print("Не удалось собрать данные.")
        return  # Если данных нет, прерываем выполнение функции

    # ---------------------------

    if all_raw_matches:
        new_df = pd.DataFrame(all_raw_matches)

        if "match_id" in new_df.columns:
            new_df = new_df.set_index("match_id")

        # Проверяем наличие df в глобальной области видимости
        df = (
            pd.concat([df, new_df], axis=0)
            if ("df" in globals() and not df.empty)
            else new_df
        )

        df = df[~df.index.duplicated(keep="first")]
        df = df.apply(clean_teams_in_row, axis=1)
        df = df[(df["game_mode"] == 22) & (df["duration"] > (15 * 60))]

        print(
            f"📊 Датасет успешно увеличен! Текущий размер: {df.shape} уникальных матчей.\n"
        )
    else:
        print("Новых матчей не собрано.")

    # ------------------------------

    # Сохраняем, используя глобальную переменную path (убедитесь, что она объявлена в ноутбуке)
    df.to_csv(path, index=True, index_label="match_id")
    print("Файл успешно сохранен на Google Диск!")


In [173]:
path = "/content/drive/MyDrive/core/df/dota2.csv"

if os.path.exists(path) and os.path.getsize(path) > 0:
    # Читаем файл и сразу ставим match_id как индекс таблицы
    df = pd.read_csv(path, index_col="match_id")
    print(f"📦 База успешно загружена с диска! Матчей: {len(df)}")
    df = df.apply(clean_teams_in_row, axis=1)
else:
    df = pd.DataFrame()
    print("🆕 База пуста, начинаем с нуля.")
    parser()

📦 База успешно загружена с диска! Матчей: 7507


In [174]:
# parser()

In [175]:
#def i():
print("=== ЭКСПРЕСС-ПРОВЕРКА ДАТАСЕТА ===")
print(f" Всего матчей в таблице: {df.shape[0]}")
print(f" Количество столбцов: {df.shape[1]}")

# 1. Проверяем индекс
is_match_id_index = df.index.name == 'match_id' or (df.index.astype(str).str.len() > 8).all()
print(f" Индекс является 'match_id': {' SUCCESS' if is_match_id_index else '❌ FAILED (Индекс сбился)'}")
print(f" Пример текущего индекса (первых 3 строки): {list(df.index[:3])}")

# 2. Проверяем отсутствие дубликатов
dup_count = df.index.duplicated().sum()
print(f" Найденные дубликаты матчей: {dup_count} ({' SUCCESS' if dup_count == 0 else '❌ FAILED'})")

# 3. Проверяем типы данных в командах
first_rad = df['radiant_team'].iloc[0] if not df.empty else None
is_clean_list = isinstance(first_rad, list) and len(first_rad) > 0 and isinstance(first_rad[0], int)
print(f" Формат героев в ячейках: {' SUCCESS (Чистые списки чисел)' if is_clean_list else '❌ FAILED (Всё еще текст)'}")

# 4. Проверяем среднее количество героев через правильный метод .apply(len)
if is_clean_list:
    rad_len = df['radiant_team'].apply(len).mean()
    dire_len = df['dire_team'].apply(len).mean()
    print(f" Среднее количество героев в матче: Radiant={rad_len:.2f}, Dire={dire_len:.2f} ({' SUCCESS' if rad_len == 5.0 and dire_len == 5.0 else '⚠️ Внимание, есть неполные составы!'})")

# 5. Проверяем фильтры режима и времени
min_duration = df['duration'].min() / 60
unique_modes = df['game_mode'].unique()
print(f" Минимальная длительность матча: {min_duration:.1f} мин ({' SUCCESS' if min_duration >= 15 else '❌ Есть слишком короткие матчи'})")
print(f" Уникальные режимы игры в базе: {unique_modes} ({' SUCCESS' if list(unique_modes) == [22] else '❌ Есть другие режимы кроме All Pick (22)'})")
print("==================================")
# i()

=== ЭКСПРЕСС-ПРОВЕРКА ДАТАСЕТА ===
 Всего матчей в таблице: 7507
 Количество столбцов: 11
 Индекс является 'match_id':  SUCCESS
 Пример текущего индекса (первых 3 строки): [8996981018, 8996978709, 8996978591]
 Найденные дубликаты матчей: 0 ( SUCCESS)
 Формат героев в ячейках:  SUCCESS (Чистые списки чисел)
 Среднее количество героев в матче: Radiant=5.00, Dire=5.00 ( SUCCESS)
 Минимальная длительность матча: 15.1 мин ( SUCCESS)
 Уникальные режимы игры в базе: [22] ( SUCCESS)


### ml

In [169]:
df

,match_seq_num,radiant_win,start_time,duration,lobby_type,game_mode,avg_rank_tier,num_rank_tier,cluster,radiant_team,dire_team
match_id,,,,,,,,,,,
8996981018,7558712650,False,1789301701,919,0,22,52,3,145,"[22, 82, 103, 37, 96]","[1, 18, 91, 48, 52]"
8996978709,7558711568,False,1789301631,954,0,22,24,1,189,"[14, 11, 136, 64, 20]","[119, 72, 12, 22, 51]"
8996978591,7558711669,False,1789301626,941,0,22,21,1,272,"[55, 34, 29, 16, 2]","[137, 68, 6, 138, 59]"
8996978567,7558711241,True,1789301621,928,7,22,55,3,182,"[11, 51, 135, 46, 85]","[47, 123, 10, 104, 25]"
8996976989,7558712671,False,1789301572,1028,7,22,33,1,142,"[77, 38, 155, 41, 135]","[98, 48, 5, 21, 11]"
...,...,...,...,...,...,...,...,...,...,...,...
8996917112,7558714100,True,1789299551,3126,7,22,31,6,182,"[36, 39, 85, 21, 5]","[107, 135, 53, 123, 34]"
8996917108,7558697458,False,1789299537,2385,7,22,54,6,182,"[29, 8, 123, 85, 120]","[54, 71, 51, 74, 119]"
8996917107,7558693145,False,1789299532,2221,7,22,61,3,413,"[40, 6, 68, 36, 49]","[120, 31, 18, 71, 39]"
